In [2]:
# ============================================================
# NAIVE BASELINE MODEL
# VERSION: ALL COUNTRIES
# ============================================================

import pandas as pd
import numpy as np
import ast

# Sklearn
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    mean_squared_error
)

In [3]:
# ============================================================
# LOAD DATA
# ============================================================

df = pd.read_csv("data_jobs.csv")

print("Original Shape:", df.shape)

Original Shape: (785741, 17)


In [4]:
# ============================================================
# REMOVE ROWS WITH MISSING COUNTRY
# ============================================================

df = df[df["job_country"].notna()].copy()

In [5]:
# ============================================================
# CREATE UNIFIED ANNUAL SALARY
# ============================================================

df["salary"] = df["salary_year_avg"]

hourly_mask = (
    df["salary"].isna() &
    df["salary_hour_avg"].notna()
)

df.loc[hourly_mask, "salary"] = (
    df.loc[hourly_mask, "salary_hour_avg"] * 40 * 52
)

In [6]:
# ============================================================
# REMOVE ROWS WITH MISSING SALARY
# ============================================================

df = df[df["salary"].notna()].copy()

print("After Removing Missing Salaries:", df.shape)

After Removing Missing Salaries: (32665, 18)


In [7]:
# ============================================================
# REMOVE SALARY OUTLIERS USING IQR
# ============================================================

Q1 = df["salary"].quantile(0.25)
Q3 = df["salary"].quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - (1.5 * IQR)
upper_bound = Q3 + (1.5 * IQR)

df = df[
    (df["salary"] >= lower_bound) &
    (df["salary"] <= upper_bound)
].copy()

print("After Removing Outliers:", df.shape)

After Removing Outliers: (32186, 18)


In [8]:
# ============================================================
# LOG TRANSFORM TARGET
# ============================================================

df["salary_log"] = np.log1p(df["salary"])

In [9]:
# ============================================================
# CLEAN job_skills COLUMN
# ============================================================

def clean_skills(skills):

    if pd.isna(skills):
        return []

    if isinstance(skills, str):

        try:
            parsed = ast.literal_eval(skills)

            if isinstance(parsed, list):
                skills = parsed
            else:
                skills = skills.split(",")

        except:
            skills = skills.split(",")

    if not isinstance(skills, list):
        return []

    cleaned = []

    for skill in skills:

        if pd.isna(skill):
            continue

        skill = str(skill).strip().lower()

        if skill != "":
            cleaned.append(skill)

    # Remove duplicates
    cleaned = list(dict.fromkeys(cleaned))

    return cleaned

df["clean_skills"] = df["job_skills"].apply(clean_skills)

In [10]:
# ============================================================
# CREATE SIMPLE FEATURE
# ============================================================

df["skill_count"] = df["clean_skills"].apply(len)


In [11]:
# ============================================================
# REMOVE RARE JOB TITLES
# ============================================================

title_counts = df["job_title_short"].value_counts()

valid_titles = title_counts[title_counts >= 10].index

df = df[df["job_title_short"].isin(valid_titles)].copy()

print("After Removing Rare Titles:", df.shape)

After Removing Rare Titles: (32186, 21)


In [12]:
# ============================================================
# SELECT BASELINE FEATURES
# ============================================================

baseline_features = [
    "job_title_short",
    "job_country",
    "job_schedule_type",
    "job_work_from_home",
    "job_no_degree_mention",
    "job_health_insurance",
    "skill_count"
]

X = df[baseline_features].copy()
y = df["salary_log"]

In [13]:
# ============================================================
# CONVERT BOOLEAN COLUMNS TO INTEGERS
# ============================================================

binary_features = [
    "job_work_from_home",
    "job_no_degree_mention",
    "job_health_insurance"
]

for col in binary_features:
    X[col] = X[col].astype("Int64")

In [14]:
# ============================================================
# TRAIN TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [15]:
# ============================================================
# FEATURE GROUPS
# ============================================================

categorical_features = [
    "job_title_short",
    "job_country",
    "job_schedule_type"
]

numeric_features = [
    "skill_count",
    "job_work_from_home",
    "job_no_degree_mention",
    "job_health_insurance"
]

In [16]:
# ============================================================
# PREPROCESSING
# ============================================================

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            categorical_transformer,
            categorical_features
        ),
        (
            "num",
            numeric_transformer,
            numeric_features
        )
    ]
)

In [17]:
# ============================================================
# MODEL
# ============================================================

model = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", LinearRegression())
])

In [18]:
# ============================================================
# TRAIN
# ============================================================

print("\nTraining Baseline Model...\n")

model.fit(X_train, y_train)

print("Training Complete!")


Training Baseline Model...

Training Complete!


In [19]:
# ============================================================
# PREDICT
# ============================================================

pred_log = model.predict(X_test)

pred_salary = np.expm1(pred_log)
actual_salary = np.expm1(y_test)


In [20]:
# ============================================================
# EVALUATE
# ============================================================

r2 = r2_score(actual_salary, pred_salary)

mae = mean_absolute_error(
    actual_salary,
    pred_salary
)

rmse = np.sqrt(
    mean_squared_error(
        actual_salary,
        pred_salary
    )
)

In [21]:
# ============================================================
# RESULTS
# ============================================================

print("\n========== BASELINE RESULTS ==========")

print(f"R² Score : {r2:.4f}")
print(f"MAE      : ${mae:,.2f}")
print(f"RMSE     : ${rmse:,.2f}")


========== BASELINE RESULTS ==========
R² Score : 0.2097
MAE      : $29,646.55
RMSE     : $37,987.54


In [22]:
# ============================================================
# SAMPLE PREDICTIONS
# ============================================================

results = pd.DataFrame({
    "Actual Salary": actual_salary.values,
    "Predicted Salary": pred_salary
})

print("\nSample Predictions:")
print(results.head(10))


Sample Predictions:
   Actual Salary  Predicted Salary
0       120000.0      85811.567227
1        85000.0     125960.092290
2       111175.0      77220.940288
3       100500.0      75911.404331
4       185000.0     117382.251712
5        98800.0      84838.791451
6       109200.0     102375.348670
7        71647.0      88791.125090
8        82500.0      88503.262120
9       100880.0      79577.901827
